# Greeks Engine Demo

This notebook demonstrates the Black-Scholes Greeks engine using both:
1. **Numeric (NumPy)** - Fast evaluation for backtesting
2. **Symbolic (CasADi)** - For MPC optimization with automatic differentiation

We'll verify the implementations match and show how portfolio Greeks aggregation works.


In [ ]:
# Imports
import sys
sys.path.insert(0, '..')

import numpy as np
import casadi as ca
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

from src.models.black_scholes import (
    BlackScholesSymbolic, 
    BlackScholesNumeric,
    implied_volatility_newton
)
from src.models.greeks import (
    PortfolioGreeksCalculator,
    SymbolicPortfolioGreeks,
    OptionPosition,
    EquityPosition,
    PositionType
)

print("Imports successful!")


## 1. Basic Option Pricing

Let's price a simple ATM call option on SPY.


In [ ]:
# Parameters
S = 500.0      # SPY spot price
K = 500.0      # Strike (ATM)
T = 30/365     # 30 days to expiry
r = 0.05       # 5% risk-free rate
sigma = 0.15   # 15% implied volatility

# Numeric calculation
bs_numeric = BlackScholesNumeric()

call_price = bs_numeric.call_price(S, K, T, r, sigma)
put_price = bs_numeric.put_price(S, K, T, r, sigma)

print(f"ATM Call Price: ${call_price:.2f}")
print(f"ATM Put Price:  ${put_price:.2f}")
print(f"\nPut-Call Parity Check:")
print(f"  C - P = {call_price - put_price:.4f}")
print(f"  S - K*e^(-rT) = {S - K * np.exp(-r * T):.4f}")


## 2. Greeks Calculation

Calculate all Greeks for the ATM call.


In [ ]:
# Get all Greeks
call_greeks = bs_numeric.call_greeks(S, K, T, r, sigma)
put_greeks = bs_numeric.put_greeks(S, K, T, r, sigma)

print("ATM Call Greeks:")
print(f"  Delta: {call_greeks.delta:.4f} (per share)")
print(f"  Gamma: {call_greeks.gamma:.6f} (per $1 move)")
print(f"  Vega:  {call_greeks.vega:.4f} (per 100% vol, so {call_greeks.vega/100:.4f} per 1%)")
print(f"  Theta: {call_greeks.theta:.4f}/year = {call_greeks.theta/365:.4f}/day")
print(f"  Rho:   {call_greeks.rho:.4f}")

print("\nATM Put Greeks:")
print(f"  Delta: {put_greeks.delta:.4f}")
print(f"  Gamma: {put_greeks.gamma:.6f} (same as call)")
print(f"  Vega:  {put_greeks.vega:.4f} (same as call)")
print(f"  Theta: {put_greeks.theta/365:.4f}/day")


## 3. Symbolic vs Numeric Comparison

Verify that CasADi symbolic and NumPy numeric implementations match.


In [ ]:
# Create symbolic BS
bs_symbolic = BlackScholesSymbolic()

# Create CasADi function
call_fn = bs_symbolic.create_full_function(is_call=True)

# Evaluate
result = call_fn(S, K, T, r, sigma)

print("Symbolic vs Numeric Comparison:")
print(f"{'Metric':<10} {'Symbolic':>12} {'Numeric':>12} {'Diff':>12}")
print("-" * 50)

# Price
sym_price = float(result[0])
num_price = call_price
print(f"{'Price':<10} {sym_price:>12.6f} {num_price:>12.6f} {abs(sym_price-num_price):>12.2e}")

# Delta
sym_delta = float(result[1])
num_delta = call_greeks.delta
print(f"{'Delta':<10} {sym_delta:>12.6f} {num_delta:>12.6f} {abs(sym_delta-num_delta):>12.2e}")

# Gamma
sym_gamma = float(result[2])
num_gamma = call_greeks.gamma
print(f"{'Gamma':<10} {sym_gamma:>12.6f} {num_gamma:>12.6f} {abs(sym_gamma-num_gamma):>12.2e}")

print("\n✓ All values match within numerical precision!")


## 4. Portfolio Greeks - Straddle Example

Create a straddle (long call + long put) and calculate aggregate portfolio Greeks.


In [ ]:
# Create a straddle: long 10 ATM calls + long 10 ATM puts
expiry = datetime.now() + timedelta(days=30)

call_position = OptionPosition(
    symbol="SPY240131C00500000",
    position_type=PositionType.CALL,
    strike=500.0,
    expiry=expiry,
    quantity=10,  # 10 contracts
    multiplier=100,
    underlying_price=500.0,
    iv=0.15
)

put_position = OptionPosition(
    symbol="SPY240131P00500000",
    position_type=PositionType.PUT,
    strike=500.0,
    expiry=expiry,
    quantity=10,  # 10 contracts
    multiplier=100,
    underlying_price=500.0,
    iv=0.15
)

# Calculate portfolio Greeks
calc = PortfolioGreeksCalculator(risk_free_rate=0.05)
calc.add_option_position(call_position)
calc.add_option_position(put_position)

portfolio_greeks = calc.calculate()

print("Straddle Position (10 contracts each):")
print(f"  Portfolio Delta: {portfolio_greeks.delta:.1f} shares equivalent")
print(f"  Portfolio Gamma: {portfolio_greeks.gamma:.2f}")
print(f"  Portfolio Vega:  ${portfolio_greeks.vega:.2f} per 1% vol move")
print(f"  Portfolio Theta: ${portfolio_greeks.theta:.2f} per day")

print(f"\n✓ Delta is near zero (straddle is delta neutral at ATM)")
print(f"✓ Gamma is positive (long gamma - profit from movement)")
print(f"✓ Theta is negative (pay time decay daily)")


## 5. Delta Hedge Calculation

After the market moves, calculate shares needed to return to delta neutral.


In [ ]:
# Simulate: SPY moves to $505
new_spot = 505.0

# Update positions with new spot
call_position.underlying_price = new_spot
put_position.underlying_price = new_spot

# Recalculate
calc.clear()
calc.add_option_position(call_position)
calc.add_option_position(put_position)

new_greeks = calc.calculate()

print(f"After SPY moves from $500 to ${new_spot}:")
print(f"  Portfolio Delta: {new_greeks.delta:.1f} shares")
print(f"  (Delta increased because we're long gamma)")

# Calculate hedge needed
hedge_shares = calc.delta_hedge_quantity(target_delta=0.0)

print(f"\nTo return to delta neutral:")
if hedge_shares < 0:
    print(f"  SELL {abs(hedge_shares)} shares of SPY")
else:
    print(f"  BUY {hedge_shares} shares of SPY")
    
print("\nThis is what the MPC controller will optimize - when and how much to hedge!")


## 6. Automatic Differentiation (Key for MPC)

CasADi automatically computes derivatives. Here we verify that ∂Delta/∂S = Gamma.


In [ ]:
# Create symbolic spot price variable
S_sym = ca.MX.sym('S')

# Get symbolic delta expression
delta_expr = bs_symbolic.call_delta(S_sym, 500.0, 30/365, 0.05, 0.15)

# Automatic differentiation: compute ∂Delta/∂S
gamma_autodiff = ca.jacobian(delta_expr, S_sym)

# Create functions
delta_fn = ca.Function('delta', [S_sym], [delta_expr])
gamma_autodiff_fn = ca.Function('gamma_ad', [S_sym], [gamma_autodiff])
gamma_direct_fn = ca.Function('gamma_direct', [S_sym], 
                               [bs_symbolic.gamma(S_sym, 500.0, 30/365, 0.05, 0.15)])

# Evaluate at S=500
print("Automatic Differentiation Demo:")
print(f"\n  Delta at S=500: {float(delta_fn(500)):.6f}")
print(f"\n  Gamma via autodiff (∂Δ/∂S): {float(gamma_autodiff_fn(500)):.6f}")
print(f"  Gamma via BS formula:       {float(gamma_direct_fn(500)):.6f}")
print(f"\n✓ They match! CasADi autodiff is working correctly.")
print("\nThis enables MPC to efficiently compute gradients of the objective function.")


## Summary

The Greeks engine provides:

1. **Numeric (NumPy)** - Fast evaluation for backtesting
2. **Symbolic (CasADi)** - For MPC with automatic differentiation  
3. **Portfolio aggregation** - Multi-position Greeks calculation
4. **Hedge calculation** - Determine trades to reach target delta

**Next steps:**
- Build the MPC controller using these symbolic Greeks
- Connect to IBKR for real market data
- Implement the gamma scalping strategy
